In [211]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [212]:

df = pd.read_csv(
    '/content/Resume.csv',
    engine='python',          # handles complex text
    on_bad_lines='skip',      # skips corrupted rows
    encoding='utf-8'
)

In [213]:
df.head()

,ID,Resume_str,Resume_html,Category
0,16852973,HR ADMINISTRATOR/MARKETING ASSOCIATE\...,"<div class=""fontsize fontface vmargins hmargin...",HR
1,22323967,"HR SPECIALIST, US HR OPERATIONS ...","<div class=""fontsize fontface vmargins hmargin...",HR
2,33176873,HR DIRECTOR Summary Over 2...,"<div class=""fontsize fontface vmargins hmargin...",HR
3,27018550,HR SPECIALIST Summary Dedica...,"<div class=""fontsize fontface vmargins hmargin...",HR
4,17812897,HR MANAGER Skill Highlights ...,"<div class=""fontsize fontface vmargins hmargin...",HR


In [214]:
df.shape

(2484, 4)

In [215]:
df.isnull().sum()

,0
ID,0
Resume_str,0
Resume_html,0
Category,0


In [216]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2484 entries, 0 to 2483
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype 
---  ------       --------------  ----- 
 0   ID           2484 non-null   int64 
 1   Resume_str   2484 non-null   object
 2   Resume_html  2484 non-null   object
 3   Category     2484 non-null   object
dtypes: int64(1), object(3)
memory usage: 77.8+ KB


In [217]:
df['Category'].value_counts()

,count
Category,
INFORMATION-TECHNOLOGY,120
BUSINESS-DEVELOPMENT,120
ADVOCATE,118
CHEF,118
ENGINEERING,118
ACCOUNTANT,118
FINANCE,118
FITNESS,117
AVIATION,117


In [218]:
df[['Resume_str', 'Resume_html', 'Category']].sample(2)

,Resume_str,Resume_html,Category
2125,GENERAL MANAGER Professional ...,"<div class=""fontsize fontface vmargins hmargin...",PUBLIC-RELATIONS
2133,TRANSFER RECRUITER/ADMISSIONS COUNSEL...,"<div class=""fontsize fontface vmargins hmargin...",PUBLIC-RELATIONS


In [219]:
#converting all text into lowercase
df.columns = df.columns.str.lower()
df['resume_str']=df['resume_str'].str.lower()
df['category']=df['category'].str.lower()
df['resume_html']=df['resume_str'].str.lower()

In [220]:
df.head()

,id,resume_str,resume_html,category
0,16852973,hr administrator/marketing associate\...,hr administrator/marketing associate\...,hr
1,22323967,"hr specialist, us hr operations ...","hr specialist, us hr operations ...",hr
2,33176873,hr director summary over 2...,hr director summary over 2...,hr
3,27018550,hr specialist summary dedica...,hr specialist summary dedica...,hr
4,17812897,hr manager skill highlights ...,hr manager skill highlights ...,hr


In [221]:
def clean_resume(text):
    text = re.sub(r'http\S+|www\S+', '', text)      # remove URLs
    text = re.sub(r'\S+@\S+', '', text)            # remove emails
    text = re.sub(r'\d+', '', text)                # remove numbers
    text = re.sub(r'[^a-zA-Z\s]', '', text)        # remove special characters
    text = text.lower()                            # lowercase
    text = text.strip()                            # remove extra spaces
    return text

df['resume_str'] = df['resume_str'].apply(clean_resume)

df['resume_html'] = df['resume_html'].apply(clean_resume)


In [222]:
from bs4 import BeautifulSoup

def remove_html_tags(text):
    soup = BeautifulSoup(text, "html.parser")
    return soup.get_text()

df['resume_html'] = df['resume_html'].apply(remove_html_tags)


In [223]:
df.head()


,id,resume_str,resume_html,category
0,16852973,hr administratormarketing associate\n\nhr admi...,hr administratormarketing associate\n\nhr admi...,hr
1,22323967,hr specialist us hr operations summary ...,hr specialist us hr operations summary ...,hr
2,33176873,hr director summary over years exp...,hr director summary over years exp...,hr
3,27018550,hr specialist summary dedicated drive...,hr specialist summary dedicated drive...,hr
4,17812897,hr manager skill highlights ...,hr manager skill highlights ...,hr


In [224]:
df[['resume_str','resume_html']].sample(5)

,resume_str,resume_html
1670,key holder sales planner summary art ...,key holder sales planner summary art ...
1696,engineering technician professional summ...,engineering technician professional summ...
54,hr specialist summary resultsdriven p...,hr specialist summary resultsdriven p...
2090,about creative communications professional ...,about creative communications professional ...
269,information technology specialist ex...,information technology specialist ex...


In [225]:
#both are exactly same
(df['resume_str'] == df['resume_html']).sum()

np.int64(2484)

In [226]:
df = df.drop(columns=['resume_html','id'])

In [227]:
df.head()

,resume_str,category
0,hr administratormarketing associate\n\nhr admi...,hr
1,hr specialist us hr operations summary ...,hr
2,hr director summary over years exp...,hr
3,hr specialist summary dedicated drive...,hr
4,hr manager skill highlights ...,hr


In [228]:
df['resume_str'][0]

'hr administratormarketing associate\n\nhr administrator       summary     dedicated customer service manager with  years of experience in hospitality and customer service management   respected builder and leader of customerfocused teams strives to instill a shared enthusiastic commitment to customer service         highlights         focused on customer satisfaction  team management  marketing savvy  conflict resolution techniques     training and development  skilled multitasker  client relations specialist           accomplishments      missouri dot supervisor training certification  certified by ihg in customer loyalty and marketing by segment   hilton worldwide general manager training certification  accomplished trainer for cross server hospitality systems such as    hilton onq     micros    opera pms    fidelio    opera    reservation system ors    holidex    completed courses and seminars in customer service sales strategies inventory control loss prevention safety time manage

In [229]:
import re

def clean_text_advanced(text):
    text = str(text)

    # 1. Replace newline characters with space
    text = re.sub(r'\n+', ' ', text)

    # 2. Add space between joined words (lowercase + uppercase OR word boundaries)
    text = re.sub(r'([a-z])([A-Z])', r'\1 \2', text)

    # 3. Fix merged common words (custom fixes if needed)
    text = re.sub(r'administratormarketing', 'administrator marketing', text)

    # 4. Remove special characters like ,,,
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)

    # 5. Remove extra spaces
    text = re.sub(r'\s+', ' ', text)

    # 6. Lowercase
    text = text.lower().strip()


    return text

df['resume_str'] = df['resume_str'].apply(clean_text_advanced)

In [230]:
from nltk.stem import PorterStemmer
stemmer=PorterStemmer()
df['tokens'] = df['resume_str'].apply(lambda x: x.split())
df['tokens'] = df['tokens'].apply(
    lambda words: [stemmer.stem(word) for word in words]
)
df['resume_str'] = df['tokens'].apply(lambda x: " ".join(x))

In [231]:
df.head()

,resume_str,category,tokens
0,hr administr market associ hr administr summar...,hr,"[hr, administr, market, associ, hr, administr,..."
1,hr specialist us hr oper summari versatil medi...,hr,"[hr, specialist, us, hr, oper, summari, versat..."
2,hr director summari over year experi in recrui...,hr,"[hr, director, summari, over, year, experi, in..."
3,hr specialist summari dedic driven and dynam w...,hr,"[hr, specialist, summari, dedic, driven, and, ..."
4,hr manag skill highlight hr skill hr depart st...,hr,"[hr, manag, skill, highlight, hr, skill, hr, d..."


In [232]:
df['category'].value_counts()

,count
category,
information-technology,120
business-development,120
advocate,118
chef,118
engineering,118
accountant,118
finance,118
fitness,117
aviation,117


words into categorical **values**

In [233]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['category']=le.fit_transform(df['category'])

In [234]:
df.head()

,resume_str,category,tokens
0,hr administr market associ hr administr summar...,19,"[hr, administr, market, associ, hr, administr,..."
1,hr specialist us hr oper summari versatil medi...,19,"[hr, specialist, us, hr, oper, summari, versat..."
2,hr director summari over year experi in recrui...,19,"[hr, director, summari, over, year, experi, in..."
3,hr specialist summari dedic driven and dynam w...,19,"[hr, specialist, summari, dedic, driven, and, ..."
4,hr manag skill highlight hr skill hr depart st...,19,"[hr, manag, skill, highlight, hr, skill, hr, d..."


In [235]:
df.category.unique()

array([19, 13, 20, 23,  1,  9, 18, 17,  2,  8, 22, 12, 14,  5, 10, 16,  3,
       15,  0, 11, 21,  7,  4,  6])

In [236]:
#vecotrization


from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(max_features=5000)
X = tfidf.fit_transform(df['resume_str'])

In [237]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, df['category'], test_size=0.2, random_state=42
)


In [238]:
X_train.shape

(1987, 5000)

In [239]:

X_test.shape

(497, 5000)

Now let’s train the model and print the classification report:

In [246]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Train KNN (NO OneVsRest needed)
knn_model = KNeighborsClassifier(n_neighbors=5, metric='cosine')

knn_model.fit(X_train, y_train)

# Predict
y_pred_knn = knn_model.predict(X_test)

# Evaluation
print("\nKNN Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_knn):.4f}")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred_knn))
print("Classification Report:\n", classification_report(y_test, y_pred_knn))


KNN Results:
Accuracy: 0.4849
Confusion Matrix:
 [[25  0  0  0  1  0  0  0  0  0  0  0  1  0  0  0  2  0  0  0  0  0  0  0]
 [ 2 12  1  1  0  0  0  0  0  0  0  0  3  0  1  0  0  1  7  0  0  0  0  2]
 [ 2  0  2  0  0  0  0  0  0  0  0  0  1  0  0  1  0  0  0  0  0  0  0  2]
 [ 0  5  0  6  0  2  0  0  0  1  0  1  0  1  0  1  0  0  1  0  0  0  0  2]
 [ 0  3  0  1  0  0  0  0  0  0  0  0  4  2  0  0  0  0  0  1  0  0  1  6]
 [ 0  2  0  0  1  1  1  0  0  0  0  0  0  0  0  0  1  0  0  0  0  0  0  0]
 [ 0  0  0  2  0  0 13  0  0  0  0  0  0  1  0  1  0  0  0  1  3  0  0  0]
 [ 1  0  0  0  0  0  0 11  1  1  0  0  3  0  0  1  3  0  0  0  1  0  1  0]
 [ 0  1  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  0  1  0  0  0]
 [ 0  2  1  3  0  0  0  2  1 14  0  0  1  0  0  0  0  0  3  0  0  0  0  0]
 [ 0  2  0  1  0  0  3  0  0  0 15  0  1  0  0  0  0  0  1  0  0  0  1  0]
 [ 0  2  1  4  2  0  0  0  0  0  0 22  1  0  0  2  0  0  0  0  0  0  0  0]
 [ 0  1  1  1  0  0  1  0  1  0  0  1  4  1  2  0 

In [247]:
svm_model = SVC(kernel='linear')

svm_model.fit(X_train, y_train)
y_pred_svm = svm_model.predict(X_test)

print("\nSVM Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_svm):.4f}")


SVM Results:
Accuracy: 0.6036


In [248]:
lr_model = LogisticRegression(max_iter=1000)

lr_model.fit(X_train, y_train)
y_pred_lr = lr_model.predict(X_test)

print("\nLogistic Regression Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_lr):.4f}")


Logistic Regression Results:
Accuracy: 0.6177


In [250]:
rf_model = RandomForestClassifier(n_estimators=100)

rf_model.fit(X_train, y_train)
y_pred_rf = rf_model.predict(X_test)

print("\nRandom Forest Results:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf):.4f}")


Random Forest Results:
Accuracy: 0.6539


In [255]:
results = {
    "KNN": accuracy_score(y_test, y_pred_knn),
    "SVM": accuracy_score(y_test, y_pred_svm),
    "Logistic Regression": accuracy_score(y_test, y_pred_lr),
    "Random Forest": accuracy_score(y_test, y_pred_rf)

}

for model, acc in results.items():
    print(f"{model}: {acc:.4f}")

KNN: 0.4849
SVM: 0.6036
Logistic Regression: 0.6177
Random Forest: 0.6539



Prediction System

In [256]:
def predict_resume_category(input_resume, model, tfidf, label_encoder):
    """
    Predict the category of a resume
    """

    try:
        # 1. Safety check
        if not input_resume or not isinstance(input_resume, str):
            return "Invalid input"

        # 2. Clean text
        cleaned_text = clean_resume(input_resume)

        # 3. Vectorize
        vectorized_text = tfidf.transform([cleaned_text])

        # 4. Predict
        prediction = model.predict(vectorized_text)

        # 5. Decode label
        category = label_encoder.inverse_transform(prediction)

        return category[0]

    except Exception as e:
        return f"Error: {str(e)}"

In [257]:
predict_resume_category("HR manager with 5 years experience", knn_model, tfidf, le)

'hr'

In [267]:
predict_resume_category(
    "england at it doing nothing for 5 years experience",
    rf_model,
    tfidf,
    le
)

'business-development'